In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from tensorflow import keras
from scipy.stats import multivariate_normal as normal
import munch
import pylab
import sys
import os

date_str = "20260926"
output_folder_name = f'png_elu_{date_str}'

os.makedirs('logs_elu', exist_ok=True)
os.makedirs(f'{output_folder_name}', exist_ok=True)
os.makedirs('figs_elu', exist_ok=True)

In [ ]:
"""
Standalone loader for _grad.keras models
No need to import solver.py or any server-side code; rebuilds the inference model directly
"""

import zipfile, tempfile, os
import h5py
import numpy as np
import tensorflow as tf


def load_deepnn_grad(keras_path,
                     input_dim=2,
                     hidden_sizes=(100, 100, 100),
                     output_dim=2,
                     activation="elu"):
    """
    Load the DeepNN (critic_grad version) from a .keras file.

    Args:
        keras_path:    path to the .keras file
        input_dim:     input dimension (default 2, i.e. dim=2)
        hidden_sizes:  number of units per hidden layer (default (100,100,100), i.e. num_hiddens_critic)
        output_dim:    output dimension (equals input_dim for critic_grad)
        activation:    activation function (default "elu")

    Returns:
        A Keras Model that can be called directly as model(x) for inference
    """
    # 0. Keep the global float precision consistent with training
    tf.keras.backend.set_floatx("float64")

    # 1. Rebuild an identical network by hand
    inputs = tf.keras.Input(shape=(input_dim,), dtype=tf.float64)
    x = inputs
    dense_layers = []

    # Hidden layers: Dense(use_bias=False) + ELU
    for h in hidden_sizes:
        layer = tf.keras.layers.Dense(h, use_bias=False, activation=None, dtype=tf.float64)
        dense_layers.append(layer)
        x = layer(x)
        if activation == "elu":
            x = tf.nn.elu(x)
        elif activation == "relu":
            x = tf.nn.relu(x)
        elif activation == "leaky":
            x = tf.nn.leaky_relu(x)
        elif activation == "sigmoid":
            x = x + tf.math.sigmoid(x) - 0.5

    # Output layer: Dense(output_dim, use_bias=False) -- the critic_grad structure
    out_layer = tf.keras.layers.Dense(output_dim, use_bias=False, activation=None, dtype=tf.float64)
    dense_layers.append(out_layer)
    outputs = out_layer(x)

    model = tf.keras.Model(inputs, outputs)

    # 2. Open the .keras file (a zip archive) and extract model.weights.h5
    with zipfile.ZipFile(keras_path) as z:
        with tempfile.TemporaryDirectory() as tmp:
            z.extract("model.weights.h5", tmp)
            h5_path = os.path.join(tmp, "model.weights.h5")

            with h5py.File(h5_path, "r") as f:
                # Read the kernels in the order dense / dense_1 / dense_2 / ...
                n = len(dense_layers)
                keys = ["dense_layers/dense/vars/0"] + \
                       [f"dense_layers/dense_{i}/vars/0" for i in range(1, n)]

                for layer, key in zip(dense_layers, keys):
                    w = np.array(f[key])
                    assert w.shape == tuple(layer.kernel.shape), \
                        f"Shape mismatch for {key}: weights {w.shape} vs layer {layer.kernel.shape}"
                    layer.kernel.assign(w)

    return model


# plot drift control

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import os

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from tensorflow import keras
from scipy.stats import multivariate_normal as normal
import munch
import pylab
import sys
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import re

m1, m2, m3, m4 = 1.0*20/19, 2.0*20/19, 1.0*20/19, 2.0*20/19  # average service time in limiting system
M_tilde_matrix = np.array([[m1, m4], [m2, m3]])
H = np.array([[120.0/19, 0],[0, 120.0/19]], dtype=float)
gradient_to_drift_matrix = M_tilde_matrix @ np.linalg.inv(H) @ M_tilde_matrix.T

# configure fonts and styles
plt.rcParams.update({'font.size': 28, 'text.usetex': True})
font1 = {'family': 'Arial', 'weight': 'normal', 'size': 28}

# parameterized configuration
SAVE_DIR = output_folder_name
MODEL_DIR = "../../../1_NN_training/3_RS_saved_NNs"
PLOT_SHAPE = (1, 2)
FIG_SIZE = (14, 7)

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)  # create folder
    print(f"Folder '{SAVE_DIR}' created!")
else:
    print(f"Folder '{SAVE_DIR}' already exists!")

def load_models(file_name):
    """load the model and gradient-related models"""
    model_path = f"{MODEL_DIR}/{file_name}_grad.keras"
    model = load_deepnn_grad(model_path)
    cur_model_grad = model
    return model, cur_model_grad

def compute_grad_field(model, beta, rho):
    """compute the gradient field (V_gradient)"""
    x1 = np.linspace(-beta/3, (beta * 3) * rho + beta/3, 100, endpoint=True)
    x2 = np.linspace(-beta/3, (beta * 3) * rho + beta/3, 100, endpoint=True)
    x1_mat, x2_mat = np.meshgrid(x1, x2)
    x0 = np.stack((x1_mat.reshape([-1]), x2_mat.reshape([-1])), axis=1)
    V_gradient = model(x0).numpy()

    drift_control = -V_gradient @ gradient_to_drift_matrix.T * 0.5
    # decompose the gradient into two components
    V_gradient_0_mat = drift_control[:, 0].reshape([x2.shape[0], x1.shape[0]])
    V_gradient_1_mat = drift_control[:, 1].reshape([x2.shape[0], x1.shape[0]])

    return x1_mat, x2_mat, V_gradient_0_mat, V_gradient_1_mat

def plot_fill_between(t, y0, y1, y2, y3):
    """draw the fill region"""
    plt.fill_between(t, y0, y1, facecolor='white', alpha=1)
    plt.fill_between(t, y2, y3, facecolor='white', alpha=1)

def plot_contour(x1_mat, x2_mat, V_gradient_mat, title, font):
    """draw the contour plot"""
    CS = plt.contourf(x1_mat, x2_mat, V_gradient_mat, np.linspace(-v1/2, v1/20, 12, endpoint = True), cmap='jet_r')
    plt.title(title, fontsize=28)
    plt.xlabel(r'$w_1$', font)
    plt.ylabel(r'$w_2$', font)
    cbar = plt.colorbar()
    cbar.ax.invert_yaxis() 


def plot_drift_control(original_file_name, beta, rho_list, theta_list):
    for rho in rho_list:
        for theta in theta_list:
            # generate filename and load model
            file_name = original_file_name
            model, _ = load_models(file_name)

            # compute gradient field
            x1_mat, x2_mat, V_grad_0, V_grad_1 = compute_grad_field(model, beta, rho)

            # compute fill-region boundary
            t = np.linspace(0, (beta * 3) * rho, 100, endpoint = True)
            y0 = np.ones(t.shape) * 0
            y1 = np.maximum(0, 2.0 * t - (beta * 4) * rho)
            y2 = np.minimum(0.5 * t + (beta * 2) * rho, (beta * 3) * rho)
            y3 = np.ones(t.shape) * (beta * 3) * rho

            # create figure
            fig, axes = plt.subplots(*PLOT_SHAPE, figsize=FIG_SIZE)

            # draw the first subplot
            plt.subplot(1,2,1)
            plot_contour(x1_mat, x2_mat, V_grad_0, r'Drift control along $w_1$', font1)
            plot_fill_between(t, y0, y1, y2, y3)
            plt.xlim([0, (beta * 3) * rho])
            plt.ylim([0, (beta * 3) * rho])
            plt.xticks([0, 2, 4, 6])
            plt.yticks([0, 2, 4, 6])

            # draw the second subplot
            plt.subplot(1,2,2)
            plot_contour(x1_mat, x2_mat, V_grad_1, r'Drift control along $w_2$', font1)
            plot_fill_between(t, y0, y1, y2, y3)
            plt.xlim([0, (beta * 3) * rho])
            plt.ylim([0, (beta * 3) * rho])
            plt.xticks([0, 2, 4, 6])
            plt.yticks([0, 2, 4, 6])

            # save image
            fig.tight_layout()
            save_path = f"{SAVE_DIR}/RS_proposed_drift_{date_str}.png"
            plt.savefig(save_path, bbox_inches='tight', pad_inches=0.05)
            # plt.close(fig)

beta = 20.0/9
v1 = 1.0

for rho in ["1.0"]:
    for theta in ["-0.9"]:
    
        file_name = f"RybkoStolyar_none_100_{theta}_0.0_0.01_100_0.1_64_6000_dim=2_TD1_elu_x0=1.0_v1=1.0_v2=2.5_addcost=0.05_0.2_lr0.0005_eta={rho}"
        plot_drift_control(file_name, beta, [float(rho)], [float(theta)])

# plot singular control

In [ ]:
plt.rcParams.update({'font.size': 25, 'text.usetex': True})
font1 = {'family': 'Arial', 'weight': 'normal', 'size': 25}

# global configuration
ARROW_PROPS = [
    {"start": (-1.8, 3.5), "end": (-0.2, 3.5), "color": "red", "label": r"$L_1$"},
    {"start": (3.5, -1.8), "end": (3.5, -0.2), "color": "blue", "label": r"$L_2$"},
    {"start": (5.1 + 0.8, 6.8 + 1.6), "end": (5.1, 6.8), "color": "green", "label": r"$U_1$"},
    {"start": (6.5 + 1.6, 3.5 + 0.8), "end": (6.5, 3.5), "color": "darkorange", "label": r"$U_2$"},
]


def configure_plot():
    """global plotting configuration"""
    plt.rcParams.update({"font.size": font1["size"], "text.usetex": True})
    np.set_printoptions(precision=6, suppress=True, threshold=np.inf)


def plot_fill_between_singular(beta, rho, eta2, eta3):
    """draw the masking region"""
    t = np.arange((beta * 3) * rho * (1 - eta2), (beta * 3) * rho * eta2, 1 / 100)
    yt1 = np.maximum(0, 2.0 * t - (beta * 4) * rho * eta3)
    yt2 = np.minimum(0.5 * t + (beta * 2) * rho, (beta * 3) * rho)
    plt.fill_between(t, yt1, yt2, facecolor=[0.9,0.9,0.9], zorder=2)


def compute_grad_points(cur_model_grad, beta, rho, eta2, eta3):
    """compute the gradient field (point set)"""
    steps = 4000
    xlist = np.linspace(-beta/3, 3 * beta * rho + beta/3, steps)
    ylist = np.linspace(-beta/3, 3 * beta * rho + beta/3, steps)
    X, Y = np.meshgrid(xlist, ylist)
    X_f, Y_f = X.flatten(), Y.flatten()

    pos = ((X_f < (beta*3)*rho) 
           & (X_f > 0)
           & (Y_f < (beta*3)*rho) 
           & (Y_f > 0)
           & (Y_f > 2*X_f - (beta*4)*rho)
           & (X_f > 2*Y_f - (beta*4)*rho))
    pos2 = ((X_f < (beta*3)*rho*eta2) 
           & (X_f > (beta*3)*rho*(1-eta2))
           & (Y_f < (beta*3)*rho*eta2) 
           & (Y_f > (beta*3)*rho*(1-eta2))
           & (Y_f > 2*X_f - (beta*4)*rho*eta3)
           & (X_f > 2*Y_f - (beta*4)*rho*eta3))

    X_f = X_f[pos & (~pos2)]
    Y_f = Y_f[pos & (~pos2)]

    points = np.column_stack((X_f, Y_f))
    cur_grad = cur_model_grad(points)
    Z0, Z1 = cur_grad[:, 0].numpy(), cur_grad[:, 1].numpy()

    # classify point sets
    pos_F2 = (
        ((-m1) * Z0 + (-m2) * Z1 + outsourcing_cost_rate_limiting[0])
        / (sigma_F2[0] * (-m1) + sigma_F2[1] * (-m2))
        > Z0 / sigma_F2[0]
    )

    pos_F3 = (
        ((-m2) * Z0 + (-m1) * Z1 + outsourcing_cost_rate_limiting[1])
        / (sigma_F3[0] * (-m2) + sigma_F3[1] * (-m1))
        >
        ((-m1) * Z0 + (-m2) * Z1 + outsourcing_cost_rate_limiting[0])
        / (sigma_F3[0] * (-m1) + sigma_F3[1] * (-m2))
    )

    pos_F4 = (
        ((-m2) * Z0 + (-m1) * Z1 + outsourcing_cost_rate_limiting[1])
        / (sigma_F4[0] * (-m2) + sigma_F4[1] * (-m1))
        >
        ((-m1) * Z0 + (-m2) * Z1 + outsourcing_cost_rate_limiting[0])
        / (sigma_F4[0] * (-m1) + sigma_F4[1] * (-m2))
    )

    pos_F5 = (
        ((-m2) * Z0 + (-m1) * Z1 + outsourcing_cost_rate_limiting[1])
        / (sigma_F5[0] * (-m2) + sigma_F5[1] * (-m1))
        > Z1 / sigma_F5[1]
    )

    mask_I1 = (
        (pos_F2 & (X_f < 2 * Y_f - 4 * beta * rho * eta3))
        | (X_f < 3 * beta * rho * (1 - eta2))
    )

    mask_I2 = (
        (pos_F5 & (Y_f < 2 * X_f - 4 * beta * rho * eta3))
        | (Y_f < 3 * beta * rho * (1 - eta2))
    )

    mask_U1 = (
        ((~pos_F2) & (X_f < 2 * Y_f - 4 * beta * rho * eta3))
        | (pos_F3 & (Y_f > 3 * beta * rho * eta2))
        | (pos_F4 & (X_f > 3 * beta * rho * eta2))
    )

    mask_U2 = (
        ((~pos_F5) & (Y_f < 2 * X_f - 4 * beta * rho * eta3))
        | ((~pos_F3) & (Y_f > 3 * beta * rho * eta2))
        | ((~pos_F4) & (X_f > 3 * beta * rho * eta2))
    )

    return {
        "I1": (X_f[mask_I1], Y_f[mask_I1]),
        "I2": (X_f[mask_I2], Y_f[mask_I2]),
        "U1": (X_f[mask_U1], Y_f[mask_U1]),
        "U2": (X_f[mask_U2], Y_f[mask_U2]),
    }


def plot_singular_control(original_file_name, beta, rho_list, theta_list):
    eta2, eta3 = 0.996, 0.994
    configure_plot()

    for rho in rho_list:
        for theta in theta_list:
            # filename generation and model loading
            file_name = original_file_name
            _, cur_model_grad = load_models(file_name)

            # plotting initialization
            plt.figure(figsize=[7, 7])
            plot_fill_between_singular(beta, rho, eta2, eta3)

            # compute gradient points and draw
            grad_points = compute_grad_points(cur_model_grad, beta, rho, eta2, eta3)
            plt.plot(*grad_points["I1"], ".r", markersize=1, alpha=1, label=r"$I_1$")
            plt.plot(*grad_points["I2"], ".b", markersize=1, alpha=1, label=r"$I_2$")
            plt.plot(*grad_points["U1"], ".g", markersize=1, alpha=1, label=r"$U_1$")
            plt.plot(*grad_points["U2"], ".", color="darkorange", markersize=1, alpha=1, label=r"$U_2$")

            # add arrows
            for arrow in ARROW_PROPS:
                plt.annotate(
                    "",
                    xy=arrow["end"],
                    xytext=arrow["start"],
                    arrowprops=dict(arrowstyle="-|>", color=arrow["color"], linewidth=3),
                )

            # custom legend
            custom_legend = [
                Patch(facecolor=[0.9,0.9,0.9], edgecolor="none", label=r"$\mathcal{W}$"),
                *[
                    Line2D(
                        [0],
                        [0],
                        color=arrow["color"],
                        marker="None",
                        linestyle="-",
                        linewidth=3,
                        label=arrow["label"],
                    )
                    for arrow in ARROW_PROPS
                ],
            ]
            plt.legend(handles=custom_legend, loc="lower right", fontsize=22)

            labels = [r'$F_1$', r'$F_2$', r'$F_3$', r'$F_4$', r'$F_5$', r'$F_6$']
            eta_tmp = 8.0/9
            positions = [
                (0.5 * eta_tmp, 2.5 * eta_tmp),  # edge 1 midpoint
                ((2.5+0.4) * eta_tmp, (6.25-0.4) * eta_tmp),  # edge 2 midpoint
                (6 * eta_tmp, (7.5-0.5) * eta_tmp),  # edge 3 midpoint
                ((7.5-0.5) * eta_tmp, 6 * eta_tmp),  # edge 4 midpoint
                ((6.25-0.4) * eta_tmp , (2.5+0.4) * eta_tmp),  # edge 5 midpoint
                (2.5 * eta_tmp, 0.5 * eta_tmp)  # point 6 (adjusted)
            ]
            for label, (x, y) in zip(labels, positions):
                plt.text(x, y, label, fontsize = 24, ha='center', va='center')


            # set up axes and save
            plt.xlim([-1.9, 3 * beta * rho + 2.5])
            plt.ylim([-1.9, 3 * beta * rho + 2])
            plt.xticks([0, 2, 4, 6, 8])
            plt.yticks([0, 2, 4, 6, 8])
            plt.xlabel(r"$w_1$", font1)
            plt.ylabel(r"$w_2$", font1)
            plt.gca().spines['top'].set_visible(False)
            plt.gca().spines['right'].set_visible(False)
            plt.gca().xaxis.set_ticks_position('bottom')
            plt.gca().yaxis.set_ticks_position('left')
            plt.tight_layout()
            plt.grid(False)
            # plt.title(r'eta = ' + str(eta) + ', theta = ' + str(theta), font1)

            save_path = f"{SAVE_DIR}/RS_proposed_singular_{date_str}.png"
            plt.savefig(save_path, bbox_inches='tight', pad_inches=0.05)
            # plt.close()



rho_list = [1.0]
theta_list = [-0.9]
m1, m2 = 20.0 / 19, 40.0 / 19
sigma_F2 = [1.0 / 2, -1.0]
sigma_F3 = [0, -1]
sigma_F4 = [-1, 0]
sigma_F5 = [-1.0, 1.0 / 2]

v1, v2 = 1.0, 2.5
outsourcing_cost_rate_limiting = [v1, v2]


for rho in ["1.0"]:
    for theta in ["-0.9"]:
    
        file_name = f"RybkoStolyar_none_100_{theta}_0.0_0.01_100_0.1_64_6000_dim=2_TD1_elu_x0=1.0_v1=1.0_v2=2.5_addcost=0.05_0.2_lr0.0005_eta={rho}"
        plot_singular_control(file_name, beta, [float(rho)], [float(theta)])

